In [54]:
import requests
import os
import json
import pandas as pd
from tqdm import tqdm
import concurrent.futures
from requests.adapters import HTTPAdapter, Retry
import glob

In [13]:
MGNIFY_API_BASE = "https://www.ebi.ac.uk/metagenomics/api/v1"
OUTPUT_DIR = "/home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw"


In [10]:
def get_human_gut_studies():
    """Fetch all studies related to the human gut from the MGnify API by handling pagination."""
    url = f"{MGNIFY_API_BASE}/biomes/root:Host-associated:Human:Digestive system:Large intestine/studies"
    all_studies = []
    while url:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        # Extract study IDs from the current page
        all_studies.extend(study["id"] for study in data["data"])
        
        # Get next page URL from the "links" field
        url = data["links"].get("next")  # Will be None if there's no next page
    print(f"found {len(all_studies)} studies!")
    return all_studies


In [39]:
def get_studies_metadata(study_id_list):
    studies_metadata = {
        "bioproject-id": [],
        "mgnify-id": [],
        "samples-count": [],
        "last-update": [],
        "secondary-accession": [],
        "centre-name": [],
        "study-abstract": [],
        "study-name": []
    }
    for mgnify_id in tqdm(study_id_list):
        study_api_url = f"{MGNIFY_API_BASE}/studies/{mgnify_id}"
        study_response = requests.get(study_api_url)
        study_response.raise_for_status()
        study_data = study_response.json()
        metadata = study_data["data"]["attributes"]
        studies_metadata["bioproject-id"].append(metadata["bioproject"])
        studies_metadata["mgnify-id"].append(metadata["accession"])
        studies_metadata["samples-count"].append(metadata["samples-count"])
        studies_metadata["last-update"].append(metadata["last-update"])
        studies_metadata["secondary-accession"].append(metadata["secondary-accession"])
        studies_metadata["centre-name"].append(metadata["centre-name"])
        studies_metadata["study-abstract"].append(metadata["study-abstract"])
        studies_metadata["study-name"].append(metadata["study-name"])
    df = pd.DataFrame.from_dict(studies_metadata, orient="columns")
    df.to_csv(f"{OUTPUT_DIR}/studies_metadata.csv")
    return

def get_session_with_retries():
    session = requests.Session()
    retries = Retry(
        total=5,                     # Retry up to 5 times
        backoff_factor=1,            # Wait 1s, 2s, 4s... between retries
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET", "POST"]  # Methods to retry on
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def fetch_samples_for_study(study_id, session):
    """Fetch all sample metadata for a given study ID."""
    samples_metadata = []
    study_samples_url = f"{MGNIFY_API_BASE}/studies/{study_id}/samples?page_size=1000"
    
    while study_samples_url:
        try:
            response = session.get(study_samples_url)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching study {study_id} at {study_samples_url}: {e}")
            break
        
        for sample in data["data"]:
            metadata_dict = {item["key"]: item["value"] for item in sample["attributes"]["sample-metadata"]}
            samples_metadata.append({
                "sample-id": sample["id"],
                "project-name": metadata_dict.get("project name", "N/A"),
                "longitude": sample["attributes"].get("longitude", "N/A"),
                "latitude": sample["attributes"].get("latitude", "N/A"),
                "location": metadata_dict.get("geographic location (country and/or sea,region)", "N/A"),
                "date": sample["attributes"].get("last-update", "N/A"),
                "sequencing-method": metadata_dict.get("sequencing method", "N/A"),
                "collection-material": sample["attributes"].get("environment-material", "N/A"),
                "description": sample["attributes"].get("sample-desc", "N/A"),
            })
        
        study_samples_url = data["links"].get("next")  # Get next page if available
    
    return samples_metadata


def get_samples_metadata(study_id_list):
    """Fetch all sample metadata for a list of studies in parallel."""
    all_samples = []
    session = get_session_with_retries()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(fetch_samples_for_study, study_id, session): study_id for study_id in study_id_list}
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Fetching Metadata"):
            try:
                result = future.result()
                all_samples.extend(result)
            except Exception as exc:
                study_id = futures[future]
                print(f"Study {study_id} generated an exception: {exc}")
    
    # Convert to DataFrame and save
    df = pd.DataFrame(all_samples).drop_duplicates(subset="sample-id")
    df.to_csv(f"{OUTPUT_DIR}/samples_metadata.csv", index=False)
    
    print(f"Saved metadata for {len(df)} samples.")
    return df  # Optional: Return DataFrame if needed

In [12]:
study_ids = get_human_gut_studies()

found 308 studies!


100%|██████████| 308/308 [30:51<00:00,  6.01s/it]


OSError: Cannot save file into a non-existent directory: '/dataset/mgnify/mgnify_raw'

In [36]:
get_studies_metadata(study_ids)

100%|██████████| 308/308 [31:22<00:00,  6.11s/it] 


In [40]:
get_samples_metadata(study_ids)

Fetching Metadata: 100%|██████████| 308/308 [48:30<00:00,  9.45s/it]  


Saved metadata for 61644 samples.


,sample-id,project-name,longitude,latitude,location,date,sequencing-method,collection-material,description
0,SRS1465262,N/A,NaN,NaN,USA,2024-05-13T20:09:07,N/A,ENVO:00002003,Keywords: GSC:MIxS MIMS:5.0
1,SRS1465274,N/A,NaN,NaN,USA,2024-05-13T20:03:59,N/A,ENVO:00002003,Keywords: GSC:MIxS MIMS:5.0
2,SRS1465271,N/A,NaN,NaN,USA,2024-05-13T20:00:59,N/A,ENVO:00002003,Keywords: GSC:MIxS MIMS:5.0
3,SRS1465258,N/A,NaN,NaN,USA,2024-05-13T19:56:37,N/A,ENVO:00002003,Keywords: GSC:MIxS MIMS:5.0
4,SRS1465248,N/A,NaN,NaN,USA,2024-05-13T19:52:44,N/A,ENVO:00002003,Keywords: GSC:MIxS MIMS:5.0
...,...,...,...,...,...,...,...,...,...
67424,ERS1133528,N/A,-0.12,51.5,United Kingdom,2017-04-06T11:31:29,Illumina Miseq,ENVO:feces,TwinsUK Fecal Sample
67425,ERS1133529,N/A,-0.12,51.5,United Kingdom,2017-04-06T11:31:29,Illumina Miseq,ENVO:feces,TwinsUK Fecal Sample
67426,ERS1133530,N/A,-0.12,51.5,United Kingdom,2017-04-06T11:31:29,Illumina Miseq,ENVO:feces,TwinsUK Fecal Sample
67427,ERS1133531,N/A,-0.12,51.5,United Kingdom,2017-04-06T11:31:29,Illumina Miseq,ENVO:feces,TwinsUK Fecal Sample


In [53]:
def download_samples_csv(studies_metadata_file, output_dir=f"{OUTPUT_DIR}/raw_taxonomy_abundances"):
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(studies_metadata_file)
    
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    failed_list = []
    # Iterate through each row using tqdm for progress
    for index, row in df.iterrows():
        # Replace these column names with the actual ones from your CSV file
        study_id = row.get("mgnify-id")
        second_accession = row.get("secondary-accession")  # or "second_accession" if that's your column name

        # Skip the row if any required information is missing
        if pd.isnull(study_id) or pd.isnull(second_accession):
            print(f"Skipping row {index} due to missing study ID or second accession.")
            continue
        # Construct the download URL
        download_link = f"{MGNIFY_API_BASE}/studies/{study_id}/pipelines/5.0/file/{second_accession}_taxonomy_abundances_SSU_v5.0.tsv"
        
        try:
            response = requests.get(download_link, stream=True, timeout=20)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Error downloading file for study {study_id} at {download_link}: {e}")
            failed_list.append(study_id)
            continue

        # Define the filename to save the TSV file
        filename = os.path.join(output_dir, f"{study_id}.tsv")
        try:
            with open(filename, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Error saving file {filename}: {e}")
    return failed_list
# for i, study_id in enumerate(study_ids):
#     print(f"Processing study: {study_id}")
#     acc = secondary_accessions[i]
#     download_link = f"{MGNIFY_API_BASE}/studies/{study_id}/pipelines/5.0/file/{acc}_taxonomy_abundances_SSU_v5.0.tsv"
#     response = requests.get(download_link, stream=True)
#     response.raise_for_status()
#     filename = f"/home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/{study_id}.tsv"
#     with open(filename, "wb") as file:
#         for chunk in response.iter_content(chunk_size=1024):
#             file.write(chunk)
#     print(f"Downloaded: {filename}")
failed_studies = download_samples_csv(f"{OUTPUT_DIR}/studies_metadata.csv")

Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00005333.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00005230.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00002061.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00000633.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00005154.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00006551.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00005023.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify_gut/mgnify_raw/raw_taxonomy_abundances/MGYS00002425.tsv
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/m

In [55]:
""" COMBINE ALL TSV FILES INTO ONE MEGA FILE!!!"""
dataset_path = f"{OUTPUT_DIR}/raw_taxonomy_abundances"
# Path to the TSV files (update this to your actual directory)
ts_files = glob.glob(f"{dataset_path}/*.tsv")

# Initialize an empty list to store transposed dataframes
dataframes = []

for file in ts_files:
    df = pd.read_csv(file, sep='\t', index_col=0)  # Read TSV, using first column as index
    df = df.T  # Transpose to make samples as rows and species as columns
    dataframes.append(df)

# Combine all dataframes
combined_df = pd.concat(dataframes, axis=0)

# Reset index and rename for clarity
combined_df = combined_df.reset_index()
combined_df
# Save the combined dataframe to a new TSV file
combined_df.to_csv(f"{OUTPUT_DIR}/combined_taxonomy_abundance.csv", sep=',', index=False)

In [57]:
combined_df.columns

Index(['index', 'sk__Archaea', 'sk__Archaea;k__;p__Crenarchaeota',
       'sk__Archaea;k__;p__Euryarchaeota',
       'sk__Archaea;k__;p__Euryarchaeota;c__;o__;f__;g__;s__methanogenic_archaeon_CH1270',
       'sk__Archaea;k__;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales',
       'sk__Archaea;k__;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae',
       'sk__Archaea;k__;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanobacterium',
       'sk__Archaea;k__;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanobrevibacter',
       'sk__Archaea;k__;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanobrevibacter;s__Methanobrevibacter_smithii',
       ...
       'sk__Bacteria;k__;p__Proteobacteria;c__Alphaproteobacteria;o__Rhizobiales;f__Brucellaceae;g__Ochrobactrum;s__Ochrobactrum_sp._Pb4',
       'sk__Bacteria;k__;p__Proteobacteria;

In [ ]:
combined_df.reset_index()

In [64]:
combined_df.rename(columns={"index": "sample_id", "#SampleID": "index"}, inplace=True)

In [66]:
# check for duplicate
combined_df.rename(columns={"study_id": "sample_id"}, inplace=True)
df_clean = combined_df[combined_df.duplicated(["sample_id"], keep=False)]
df_clean.shape

(0, 10271)

means no samples are duplicated, great news!

In [67]:
combined_df.to_csv(f"{OUTPUT_DIR}/combined_taxonomy_abundance.csv", sep=',', index=False)

In [68]:
combined_df.shape

(46406, 10271)